# Step 2.2 — ETL Cleaning (merged)
Runs every cleaning step in order on `delay`:

0. Setup + load tables into DuckDB
1. **Phantom-journey drop** — multi-day aware (grouped by `origin_date`)
2. Origin (`station_no = 1`) imputation — per-train median
3. Mid-route forward-fill within each journey
4. Drop unresolved nulls, export, and verify

**Imputation provenance:** the final delay dataset includes `is_imputed` (`TRUE` when `delay` was filled by this ETL, `FALSE` when it was observed in the input). Keep this column when splitting data, and use `is_imputed = FALSE` for evaluation/test rows if you need test targets to be based only on observed delays.

The raw `delay` table is **not** modified; cleaning steps write to derived tables.


In [1]:
import duckdb
import json
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 120)

BASE_DIR   = Path.cwd().parent
DATA_DIR   = BASE_DIR / "data_sets"
RESULT_DIR = BASE_DIR / "results"
RESULT_DIR.mkdir(exist_ok=True)
TMP_DIR    = RESULT_DIR / "duckdb_tmp"
TMP_DIR.mkdir(exist_ok=True)

con = duckdb.connect(str(RESULT_DIR / "step01_03_5.duckdb"))
con.execute("SET memory_limit='6GB'")            # lower this if your laptop has < 8 GB RAM
con.execute(f"SET temp_directory='{TMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order=false")

FINDINGS = {}          # everything that matters is collected here -> saved as JSON at the end


def q(sql):
    return con.execute(sql).df()


def save(df, name):
    df.to_csv(RESULT_DIR / name, index=False)
    print(f"  -> saved results/{name}")

In [2]:
# ## 0. Load tables into DuckDB (delay is loaded once and cached in the .duckdb file)

def load(name, fname, types=None):
    path = (DATA_DIR / fname).as_posix()
    t = f", types={types}" if types else ""
    con.execute(
        f"CREATE OR REPLACE TABLE {name} AS "
        f"SELECT * FROM read_csv('{path}', header=true{t})"
    )


load("train_details",      "train_details.csv",      {"train_no": "BIGINT"})
load("station_full_names", "station_full_names.csv")
load("schedule",           "combined_schedule.csv",  {"train_no": "BIGINT", "station_no": "INTEGER"})

# Set True if the old step02_02 notebook already overwrote `delay` inside the .duckdb file:
# the raw table is dropped and re-read from combined_delay.csv so Step 1 sees the original data.
RELOAD_RAW_DELAY = False
if RELOAD_RAW_DELAY:
    con.execute("DROP TABLE IF EXISTS delay")

already = con.execute(
    "SELECT count(*) FROM information_schema.tables WHERE table_name='delay'"
).fetchone()[0]
if not already:
    print("Loading combined_delay.csv (one-time, a few minutes)...")
    load("delay", "combined_delay.csv",
         {"date": "DATE", "station_no": "INTEGER", "delay": "DOUBLE", "train_no": "BIGINT"})

for t in ["train_details", "station_full_names", "schedule", "delay"]:
    print(t, con.execute(f"SELECT count(*) FROM {t}").fetchone()[0])

# Helper tables used repeatedly
con.execute("CREATE OR REPLACE TABLE sched_stations AS SELECT DISTINCT station_name FROM schedule")
con.execute("CREATE OR REPLACE TABLE delay_stations AS SELECT DISTINCT station_name FROM delay")

train_details 8992
station_full_names 8963
schedule 172112
delay 38428703


## Step 1 — Phantom-journey drop (multi-day aware)
A journey is identified by `(train_no, origin_date)` where `origin_date = date − (arrival_day − 1)`. Journeys with zero non-null delays are removed. Output table: `delay_step1`.

In [3]:
print("=== F. Discovering Completely Missing Multi-Day Train Journeys ===")

# Join delay with schedule to offset the station date by the arrival_day
# This calculates the true 'origin_date' for the entire sequence.
query = """
WITH journey_mapping AS (
    SELECT 
        d.train_no,
        d.station_no,
        d.delay,
        d.date AS station_date,
        s.arrival_day,
        -- Subtract the day offset to group the entire multi-day journey together
        (d.date - CAST((s.arrival_day - 1) AS INTEGER)) AS origin_date
    FROM delay d
    JOIN schedule s 
      ON d.train_no = s.train_no 
     AND d.station_no = s.station_no
)
SELECT 
    origin_date,
    train_no,
    COUNT(*) AS total_stations_in_route,
    COUNT(delay) AS non_null_delay_count
FROM journey_mapping
GROUP BY origin_date, train_no
HAVING COUNT(delay) = 0
ORDER BY origin_date, train_no
"""

missing_journeys = con.execute(query).df()

print(f"Total completely missing train journeys found: {len(missing_journeys)}\n")

if not missing_journeys.empty:
    print(missing_journeys.head(20))
    
    output_file = RESULT_DIR / "missing_train_journeys.csv"
    missing_journeys.to_csv(output_file, index=False)
    print(f"\n-> Saved missing train journeys to {output_file.as_posix()}")

=== F. Discovering Completely Missing Multi-Day Train Journeys ===
Total completely missing train journeys found: 39616

   origin_date  train_no  total_stations_in_route  non_null_delay_count
0   2025-02-05      9526                        1                     0
1   2025-02-05     15909                        5                     0
2   2025-02-06      4717                        7                     0
3   2025-02-06      9526                       18                     0
4   2025-02-06     12328                        1                     0
5   2025-02-06     14524                        1                     0
6   2025-02-06     14662                        6                     0
7   2025-02-06     15904                        5                     0
8   2025-02-06     20822                        1                     0
9   2025-02-07      1666                       14                     0
10  2025-02-07      4717                       12                     0
11  2025-02-07 

In [4]:
print("=== G. Verifying Completely Missing Journeys Against Schedule ===")

query = """
WITH schedule_counts AS (
    -- 1. Get the baseline: how many stations should this train visit?
    SELECT 
        train_no, 
        COUNT(station_no) AS schedule_station_count
    FROM schedule
    GROUP BY train_no
),
journey_mapping AS (
    -- 2. Group the delay data by the true multi-day journey (origin_date)
    SELECT 
        d.train_no,
        (d.date - CAST((s.arrival_day - 1) AS INTEGER)) AS origin_date,
        COUNT(d.station_no) AS delay_station_count,
        COUNT(d.delay) AS non_null_delay_count
    FROM delay d
    JOIN schedule s 
      ON d.train_no = s.train_no 
     AND d.station_no = s.station_no
    GROUP BY d.train_no, (d.date - CAST((s.arrival_day - 1) AS INTEGER))
)
-- 3. Join them together and filter for sequences with ZERO valid delays
SELECT 
    j.origin_date,
    j.train_no,
    s.schedule_station_count,
    j.delay_station_count,
    j.non_null_delay_count,
    CASE 
        WHEN s.schedule_station_count = j.delay_station_count THEN 'Full Route Logged (All Null)'
        ELSE 'Partial Route Logged' 
    END AS sequence_status
FROM journey_mapping j
JOIN schedule_counts s 
  ON j.train_no = s.train_no
WHERE j.non_null_delay_count = 0
ORDER BY j.origin_date, j.train_no
"""

verified_missing_journeys = con.execute(query).df()

print(f"Total completely missing train journeys found: {len(verified_missing_journeys)}\n")

if not verified_missing_journeys.empty:
    print(verified_missing_journeys.head(20))
    
    # Isolate discrepancies for review
    discrepancies = verified_missing_journeys[
        verified_missing_journeys['schedule_station_count'] != verified_missing_journeys['delay_station_count']
    ]
    
    print(f"\nJourneys where logged stations != scheduled stations: {len(discrepancies)}")
    if not discrepancies.empty:
        print(discrepancies.head(10))
    
    # Save the output
    output_file = RESULT_DIR / "verified_missing_train_journeys.csv"
    verified_missing_journeys.to_csv(output_file, index=False)
    print(f"\n-> Saved verified missing train journeys to {output_file.as_posix()}")

=== G. Verifying Completely Missing Journeys Against Schedule ===
Total completely missing train journeys found: 39616

   origin_date  train_no  schedule_station_count  delay_station_count  non_null_delay_count       sequence_status
0   2025-02-05      9526                      49                    1                     0  Partial Route Logged
1   2025-02-05     15909                      85                    5                     0  Partial Route Logged
2   2025-02-06      4717                      32                    7                     0  Partial Route Logged
3   2025-02-06      9526                      49                   18                     0  Partial Route Logged
4   2025-02-06     12328                      20                    1                     0  Partial Route Logged
5   2025-02-06     14524                      23                    1                     0  Partial Route Logged
6   2025-02-06     14662                      63                    6             

In [5]:
print("=== H. Dropping All Completely Null Phantom Journeys ===")

initial_count = con.execute("SELECT COUNT(*) FROM delay").fetchone()[0]

count_journeys_query = """
WITH schedule_counts AS (
    SELECT train_no, COUNT(station_no) AS schedule_station_count FROM schedule GROUP BY train_no
),
journey_stats AS (
    SELECT d.train_no, (d.date - CAST((s.arrival_day - 1) AS INTEGER)) AS origin_date, 
           COUNT(d.station_no) AS delay_station_count, COUNT(d.delay) AS non_null_delay_count
    FROM delay d JOIN schedule s ON d.train_no = s.train_no AND d.station_no = s.station_no
    GROUP BY d.train_no, (d.date - CAST((s.arrival_day - 1) AS INTEGER))
)
SELECT COUNT(*) FROM journey_stats j 
JOIN schedule_counts s ON j.train_no = s.train_no WHERE j.non_null_delay_count = 0;
"""
dropped_journeys = con.execute(count_journeys_query).fetchone()[0]

cleanup_query = """
CREATE OR REPLACE TABLE delay_step1 AS
WITH schedule_counts AS (
    SELECT train_no, COUNT(station_no) AS schedule_station_count FROM schedule GROUP BY train_no
),
journey_stats AS (
    SELECT d.train_no, (d.date - CAST((s.arrival_day - 1) AS INTEGER)) AS origin_date, 
           COUNT(d.station_no) AS delay_station_count, COUNT(d.delay) AS non_null_delay_count
    FROM delay d JOIN schedule s ON d.train_no = s.train_no AND d.station_no = s.station_no
    GROUP BY d.train_no, (d.date - CAST((s.arrival_day - 1) AS INTEGER))
),
phantom_all_journeys AS (
    SELECT j.train_no, j.origin_date 
    FROM journey_stats j JOIN schedule_counts s ON j.train_no = s.train_no WHERE j.non_null_delay_count = 0
)
SELECT d.*, (d.date - CAST((s.arrival_day - 1) AS INTEGER)) AS origin_date
FROM delay d
JOIN schedule s ON d.train_no = s.train_no AND d.station_no = s.station_no
LEFT JOIN phantom_all_journeys p ON d.train_no = p.train_no AND (d.date - CAST((s.arrival_day - 1) AS INTEGER)) = p.origin_date
WHERE p.train_no IS NULL;
"""
print("Filtering phantom runs into delay_step1...")
con.execute(cleanup_query)

final_count = con.execute("SELECT COUNT(*) FROM delay_step1").fetchone()[0]
dropped_rows = initial_count - final_count

orig_nulls = con.execute("SELECT COUNT(*) FROM delay WHERE delay IS NULL").fetchone()[0]
rem_nulls = con.execute("SELECT COUNT(*) FROM delay_step1 WHERE delay IS NULL").fetchone()[0]

print(f"\nTotal rows in 'delay_step1' after cleanup: {final_count:,}")
print(f"Total phantom journeys dropped: {dropped_journeys:,}")
print(f"Total rows dropped: {dropped_rows:,}")
print(f"Null delays eliminated via drop: {orig_nulls - rem_nulls:,}")
print(f"Null delays remaining to impute: {rem_nulls:,}")

FINDINGS["step02_02_phantom_journeys"] = {
    "rows_before": initial_count, "rows_after": final_count,
    "phantom_journeys_dropped": dropped_journeys, "rows_dropped": dropped_rows,
    "nulls_before": orig_nulls, "nulls_remaining": rem_nulls,
    "note": "Dropped ALL completely null journeys where non_null_delay_count == 0."
}

=== H. Dropping All Completely Null Phantom Journeys ===
Filtering phantom runs into delay_step1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Total rows in 'delay_step1' after cleanup: 36,640,796
Total phantom journeys dropped: 39,616
Total rows dropped: 1,787,907
Null delays eliminated via drop: 830,375
Null delays remaining to impute: 1,043,896


## Step 2 — Origin imputation (per-train median)

Adds `is_imputed`: `TRUE` for an origin delay filled by the per-train median (or fallback `0`), `FALSE` for an observed value. This provenance flag is carried through later steps.

In [6]:
print("\n=== STEP 2: ORIGIN IMPUTATION (CLASS-SPECIFIC MEDIAN) ===")

con.execute("""
    CREATE OR REPLACE TEMP TABLE TrainOriginMedian AS
    SELECT train_no, MEDIAN(delay) AS median_dep_delay
    FROM delay_step1 WHERE station_no = 1 AND delay IS NOT NULL GROUP BY train_no
""")

# Calculate global median for fallback
global_median_val = con.execute("SELECT MEDIAN(delay) FROM delay_step1 WHERE station_no = 1").fetchone()[0]
# Handle case where global median itself is missing (edge case safety)
global_median = global_median_val if global_median_val is not None else 0

origin_sample = q(f"""
    SELECT a.origin_date, a.date, a.train_no, a.station_name, a.delay AS raw_delay, 
           COALESCE(m.median_dep_delay, {global_median}) AS imputed_median_value
    FROM delay_step1 a
    LEFT JOIN TrainOriginMedian m ON a.train_no = m.train_no
    WHERE a.station_no = 1 AND a.delay IS NULL LIMIT 5
""")
print(f"\nGlobal Origin Median Fallback Value: {global_median}")
print("\nSample of Imputed Origin Delays (station_no = 1):")
print(origin_sample.to_string(index=False))

con.execute(f"""
    CREATE OR REPLACE TABLE delay_step2 AS
    SELECT a.origin_date, a.date, a.train_no, a.station_no, a.station_name,
           CASE
               WHEN a.station_no = 1 AND a.delay IS NULL THEN COALESCE(m.median_dep_delay, {global_median})
               ELSE a.delay
           END AS delay,
           (a.delay IS NULL AND a.station_no = 1) AS is_imputed
    FROM delay_step1 a
    LEFT JOIN TrainOriginMedian m ON a.train_no = m.train_no
""")

metrics_2 = q("""
    SELECT (SELECT COUNT(*) FROM delay_step1 WHERE station_no = 1 AND delay IS NULL) AS missing_origins_before,
           (SELECT COUNT(*) FROM delay_step2 WHERE station_no = 1 AND delay IS NULL) AS missing_origins_after,
           (SELECT COUNT(*) FROM delay_step2 WHERE delay IS NULL) AS total_remaining_nulls
""")

missing_origins_before = int(metrics_2['missing_origins_before'][0])
missing_origins_after = int(metrics_2['missing_origins_after'][0])

print("\n--- STEP 2 METRICS ---")
print(f"Origin Values Successfully Imputed: {missing_origins_before - missing_origins_after:,}")
print(f"Remaining Mid-Route Nulls for Step 3: {int(metrics_2['total_remaining_nulls'][0]):,}")

FINDINGS["step02_origin_imputation"] = {
    "origins_imputed": missing_origins_before - missing_origins_after,
    "remaining_nulls": int(metrics_2['total_remaining_nulls'][0]),
    "global_median_fallback_used": global_median
}


=== STEP 2: ORIGIN IMPUTATION (CLASS-SPECIFIC MEDIAN) ===

Global Origin Median Fallback Value: 1.0

Sample of Imputed Origin Delays (station_no = 1):
origin_date       date  train_no station_name  raw_delay  imputed_median_value
 2025-04-05 2025-04-05     68761          RIM        NaN                   0.0
 2025-04-06 2025-04-06     68761          RIM        NaN                   0.0
 2025-04-07 2025-04-07     68761          RIM        NaN                   0.0
 2025-04-08 2025-04-08     68761          RIM        NaN                   0.0
 2025-04-09 2025-04-09     68761          RIM        NaN                   0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- STEP 2 METRICS ---
Origin Values Successfully Imputed: 42,392
Remaining Mid-Route Nulls for Step 3: 1,001,504


## Step 3 — Mid-route forward-fill (within each `date`, `train_no` journey)

In [7]:
print("\n=== STEP 3: SPATIOTEMPORAL INTERPOLATION (FORWARD-FILL) ===")

con.execute("""
    CREATE OR REPLACE TABLE delay_imputed AS
    SELECT origin_date, date, train_no, station_no, station_name,
           LAST_VALUE(delay IGNORE NULLS) OVER (
               PARTITION BY origin_date, train_no 
               ORDER BY station_no 
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) AS delay,
           (is_imputed OR (delay IS NULL AND LAST_VALUE(delay IGNORE NULLS) OVER (
               PARTITION BY origin_date, train_no 
               ORDER BY station_no 
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) IS NOT NULL)) AS is_imputed
    FROM delay_step2
""")

filled_sample = q("""
    SELECT b.origin_date, b.date, b.train_no, b.station_name, b.station_no, 
           b.delay AS old_delay_state, a.delay AS forward_filled_delay, a.is_imputed
    FROM delay_step2 b
    JOIN delay_imputed a ON b.origin_date = a.origin_date AND b.train_no = a.train_no AND b.station_no = a.station_no
    WHERE b.delay IS NULL LIMIT 5
""")
print("\nSample of Forward-Filled Mid-Route Delays (station_no > 1):")
print(filled_sample.to_string(index=False))

metrics_3 = q("""
    SELECT (SELECT COUNT(*) FROM delay_step2 WHERE delay IS NULL) AS nulls_before_fill,
           (SELECT COUNT(*) FROM delay_imputed WHERE delay IS NULL) AS nulls_after_fill,
           (SELECT COUNT(*) FROM delay_imputed WHERE is_imputed) AS total_imputed_rows
""")

nulls_before = int(metrics_3['nulls_before_fill'][0])
nulls_after = int(metrics_3['nulls_after_fill'][0])

print("\n--- STEP 3 METRICS ---")
print(f"Values Successfully Interpolated: {nulls_before - nulls_after:,}")
print(f"Unresolved Nulls: {nulls_after:,}")

FINDINGS["step03_forward_fill"] = {
    "values_interpolated": nulls_before - nulls_after,
    "unresolved_nulls": nulls_after,
    "total_rows_marked_imputed": int(metrics_3["total_imputed_rows"][0]),
}


=== STEP 3: SPATIOTEMPORAL INTERPOLATION (FORWARD-FILL) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Sample of Forward-Filled Mid-Route Delays (station_no > 1):
origin_date       date  train_no station_name  station_no  old_delay_state  forward_filled_delay  is_imputed
 2025-06-24 2025-06-24      1435          MRX          11              NaN                 205.0        True
 2025-06-24 2025-06-24      1435          UMD          13              NaN                 190.0        True
 2025-06-24 2025-06-25     12115           MO           9              NaN                  43.0        True
 2025-06-24 2025-06-24     12309          DNR           3              NaN                   6.0        True
 2025-06-24 2025-06-25     12309          SFG           5              NaN                   0.0        True

--- STEP 3 METRICS ---
Values Successfully Interpolated: 990,514
Unresolved Nulls: 10,990


In [8]:
# === STEP 3.5: DROP REMAINING ORPHANED NULLS ===
# Keep only rows with a usable delay. The is_imputed provenance flag remains
# attached to each retained row, including values filled in Step 2 or Step 3.

print("\\n=== STEP 3.5: DROP REMAINING ORPHANED NULLS ===")

rows_before_orphan_drop = con.execute("SELECT COUNT(*) FROM delay_imputed").fetchone()[0]
orphans_to_drop = con.execute(
    "SELECT COUNT(*) FROM delay_imputed WHERE delay IS NULL"
).fetchone()[0]

con.execute("""
    CREATE OR REPLACE TABLE delay_imputed AS
    SELECT *
    FROM delay_imputed
    WHERE delay IS NOT NULL
""")

remaining_nulls = con.execute("SELECT COUNT(*) FROM delay_imputed WHERE delay IS NULL").fetchone()[0]
final_rows = con.execute("SELECT COUNT(*) FROM delay_imputed").fetchone()[0]
imputed_rows = con.execute("SELECT COUNT(*) FROM delay_imputed WHERE is_imputed").fetchone()[0]

print(f"Rows before orphan drop: {rows_before_orphan_drop:,}")
print(f"Orphaned null rows dropped: {orphans_to_drop:,}")
print(f"Remaining nulls after deletion: {remaining_nulls:,}")
print(f"Final clean row count: {final_rows:,}")
print(f"Rows marked as imputed: {imputed_rows:,}")

FINDINGS["step03_5_orphaned_drops"] = {
    "rows_before_drop": rows_before_orphan_drop,
    "dropped_orphans": orphans_to_drop,
    "remaining_nulls": remaining_nulls,
    "final_rows": final_rows,
    "imputed_rows": imputed_rows,
    "note": "Dropped remaining mid-route nulls while retaining is_imputed provenance."
}

\n=== STEP 3.5: DROP REMAINING ORPHANED NULLS ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows before orphan drop: 36,640,796
Orphaned null rows dropped: 10,990
Remaining nulls after deletion: 0
Final clean row count: 36,629,806
Rows marked as imputed: 1,032,906


## Step 4 — Export and cleanup

Exports `combined_delay_imputed.csv` with the `is_imputed` boolean provenance column. Unresolved null delays are dropped before export, so the CSV matches the final cleaned table.

In [9]:
print("\n=== FINAL EXPORT & CLEANUP ===")

final_rows = int(q("SELECT COUNT(*) AS c FROM delay_imputed")['c'][0])
output_file = RESULT_DIR / "combined_delay_imputed.csv"
print(f"Exporting {final_rows:,} rows to CSV (excluding origin_date)...")

# Explicitly defining columns to prevent origin_date from exporting to the CSV
con.execute(f"""
    COPY (
        SELECT date, train_no, station_no, station_name, delay, is_imputed 
        FROM delay_imputed
    ) TO '{output_file.as_posix()}' (HEADER, DELIMITER ',')
""")
print(f"  -> Successfully saved to results/{output_file.name}")

imputed_rows = int(q("SELECT COUNT(*) AS c FROM delay_imputed WHERE is_imputed")["c"][0])
FINDINGS["final_export"] = {
    "rows": final_rows,
    "imputed_rows": imputed_rows,
    "file": output_file.name,
    "provenance_column": "is_imputed",
}

con.execute("DROP TABLE delay_step1")
con.execute("DROP TABLE delay_step2")


=== FINAL EXPORT & CLEANUP ===
Exporting 36,629,806 rows to CSV (excluding origin_date)...
  -> Successfully saved to results/combined_delay_imputed.csv


In [10]:
# Save all findings (merged into any existing JSON) and release the DuckDB file
findings_file = RESULT_DIR / "step02_02_ETL_findings.json"
existing = {}
if findings_file.exists():
    try:
        existing = json.load(open(findings_file))
    except json.JSONDecodeError:
        existing = {}
existing.update(FINDINGS)
with open(findings_file, "w") as f:
    json.dump(existing, f, indent=4, default=int)
print(f"-> Findings saved to {findings_file.as_posix()}")

-> Findings saved to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/step02_02_ETL_findings.json


## Verification of cleaned outputs

In [11]:
# File paths (using the exact singular naming conventions from previous steps)
files = {
    "Train Dimension": RESULT_DIR / "dim_train_resolved.csv",
    "Station Dimension": RESULT_DIR / "dim_station_resolved.csv",
    "Schedule Fact": RESULT_DIR / "combined_schedule_imputed.csv"
}

print("=== SMALL DATASETS (PANDAS VERIFICATION) ===")
for name, path in files.items():
    if path.exists():
        df = pd.read_csv(path)
        print(f"\n--- {name} ({path.name}) ---")
        print(f"Total Rows: {len(df):,}")
        
        # Check for completeness (missing values)
        null_counts = df.isnull().sum()
        total_nulls = null_counts.sum()
        if total_nulls == 0:
            print("Missing Values: 0 (Perfect completeness)")
        else:
            print("Missing Values Detected:")
            print(null_counts[null_counts > 0].to_string())
        
        # Print basic uniqueness/categorical stats for string columns
        categorical_cols = df.select_dtypes(include=['object']).columns
        if len(categorical_cols) > 0:
            print(f"Unique Values in Categorical Fields:")
            for col in categorical_cols:
                print(f"  - {col}: {df[col].nunique():,} unique values")
    else:
        print(f"\n--- {name} ---")
        print(f"File not found: {path.name}")

print("\n=======================================================")
print("=== MASSIVE DELAY DATASET (DUCKDB VERIFICATION) ===")

delay_csv = RESULT_DIR / "combined_delay_imputed.csv"

if delay_csv.exists():
    # Connect to an in-memory DuckDB instance to query the CSV directly
    vcon = duckdb.connect()
    
    # Query basic statistical descriptions and completeness
    stats_query = f"""
    SELECT 
        COUNT(*) AS total_tuples,
        COUNT(*) FILTER (WHERE delay IS NULL) AS missing_delays,
        COUNT(*) FILTER (WHERE is_imputed) AS imputed_rows,
        COUNT(DISTINCT train_no) AS unique_trains,
        COUNT(DISTINCT station_no) AS unique_station_sequences,
        ROUND(AVG(delay), 2) AS mean_delay,
        ROUND(STDDEV(delay), 2) AS stddev_delay,
        MIN(delay) AS min_delay,
        MAX(delay) AS max_delay
    FROM read_csv_auto('{delay_csv.as_posix()}')
    """
    
    delay_stats = vcon.execute(stats_query).df()
    
    print(f"\n--- Delay Fact ({delay_csv.name}) ---")
    print(f"Total Rows: {int(delay_stats['total_tuples'][0]):,}")
    print(f"Missing Delay Values: {int(delay_stats['missing_delays'][0]):,}")
    print(f"Rows Marked as Imputed: {int(delay_stats['imputed_rows'][0]):,}")
    print("Provenance column: is_imputed (TRUE = filled by ETL; FALSE = observed)")
    print(f"Unique Trains Tracked: {int(delay_stats['unique_trains'][0]):,}")
    print(f"Max Station Sequence: {int(delay_stats['unique_station_sequences'][0]):,}")
    print("\nDelay Column Statistics (in minutes):")
    print(f"  - Mean:   {delay_stats['mean_delay'][0]}")
    print(f"  - StdDev: {delay_stats['stddev_delay'][0]}")
    print(f"  - Min:    {delay_stats['min_delay'][0]}")
    print(f"  - Max:    {delay_stats['max_delay'][0]}")
    vcon.close()
else:
    print(f"File not found: {delay_csv.name}")

=== SMALL DATASETS (PANDAS VERIFICATION) ===

--- Train Dimension (dim_train_resolved.csv) ---
Total Rows: 8,720
Missing Values: 0 (Perfect completeness)
Unique Values in Categorical Fields:
  - train_name: 6,455 unique values
  - primary_type_code: 8 unique values
  - raw_type_codes: 13 unique values

--- Station Dimension (dim_station_resolved.csv) ---
Total Rows: 9,027
Missing Values: 0 (Perfect completeness)
Unique Values in Categorical Fields:
  - station_name: 9,027 unique values
  - station_full_name: 8,941 unique values
  - station_zone: 20 unique values
  - station_address: 6,997 unique values

--- Schedule Fact (combined_schedule_imputed.csv) ---
Total Rows: 172,112
Missing Values: 0 (Perfect completeness)
Unique Values in Categorical Fields:
  - station_name: 8,629 unique values
  - arrival_time: 1,440 unique values
  - departure_time: 1,440 unique values

=== MASSIVE DELAY DATASET (DUCKDB VERIFICATION) ===

--- Delay Fact (combined_delay_imputed.csv) ---
Total Rows: 36,629,

In [12]:
con.close()